# 01 — Exploring the Blackjack Engine

I want the engine to disappear into the background later, so this notebook gives me a compact visual check of the rules it enforces. I use deliberately arranged shoes whenever a particular card sequence matters.

In [ ]:
from fractions import Fraction

import matplotlib.pyplot as plt

from blackjack import (
    BlackjackRound,
    Hand,
    InsuranceAction,
    PlayerAction,
    Shoe,
    calculate_hand_value,
    cards,
)

plt.style.use("seaborn-v0_8-whitegrid")

## Hand totals and soft Aces

I can treat an Ace as 11 until that would make the hand bust. If the total becomes too large, the same Ace falls back to 1. Only one Ace ever needs the extra ten points.

In [ ]:
hand_examples = {
    "hard 17": cards("10", "7"),
    "soft 17": cards("A", "6"),
    "two Aces": cards("A", "A", "8"),
    "three Aces": cards("A", "A", "A", "8"),
    "Ace falls to 1": cards("A", "9", "A", "K"),
    "bust": cards("K", "Q", "2"),
}

values = {name: calculate_hand_value(hand) for name, hand in hand_examples.items()}
for name, hand in hand_examples.items():
    value = values[name]
    labels = " ".join(str(card) for card in hand)
    kind = "soft" if value.is_soft else "hard"
    print(f"{name:16} {labels:10} → {value.total:2} ({kind})")

fig, ax = plt.subplots(figsize=(8, 3.2))
colors = ["#4c78a8" if value.is_soft else "#f58518" for value in values.values()]
ax.bar(values.keys(), [value.total for value in values.values()], color=colors)
ax.axhline(21, color="#c44e52", linestyle="--", label="21")
ax.set_ylabel("Best total")
ax.set_title("Blue hands still contain an Ace counted as 11")
ax.tick_params(axis="x", rotation=25)
ax.legend()
plt.tight_layout()
plt.show()

## A natural is narrower than “21”

I reserve the 3:2 payout for an original two-card Ace plus a ten-valued card. Three-card 21 and two-card 21 after a split are ordinary hands.

In [ ]:
twenty_ones = [
    ("original A + K", Hand(cards("A", "K"))),
    ("three-card 21", Hand(cards("7", "7", "7"))),
    ("split A + K", Hand(cards("A", "K"), from_split=True)),
]

for label, hand in twenty_ones:
    print(f"{label:16} total={hand.value.total}, natural={hand.is_natural_blackjack}")

## Dealer play: hard 17 stands, soft 17 hits

I can compare dealer paths by changing only the arranged cards. An Ace makes 17 soft while it is still counted as 11, so this ruleset takes another card.

In [ ]:
hard_17 = BlackjackRound(Shoe.arranged(cards("10", "10", "7", "7")), 10)
hard_17.act(PlayerAction.STAND)

soft_17 = BlackjackRound(Shoe.arranged(cards("10", "A", "8", "6", "10")), 10)
soft_17.decide_insurance(InsuranceAction.DECLINE)
soft_17.act(PlayerAction.STAND)

soft_path = BlackjackRound(
    Shoe.arranged(cards("10", "A", "8", "2", "A", "3", "10")), 10
)
soft_path.decide_insurance(InsuranceAction.DECLINE)
soft_path.act(PlayerAction.STAND)

dealer_examples = {
    "hard 17": hard_17.public_state.dealer_cards,
    "soft 17 hits": soft_17.public_state.dealer_cards,
    "draw through soft totals": soft_path.public_state.dealer_cards,
}

fig, ax = plt.subplots(figsize=(7.5, 3.5))
for label, dealer_cards in dealer_examples.items():
    path = [
        calculate_hand_value(dealer_cards[:index]).total
        for index in range(2, len(dealer_cards) + 1)
    ]
    ax.plot(range(2, len(dealer_cards) + 1), path, marker="o", label=label)
    print(label, "→", " ".join(str(card) for card in dealer_cards), "=", path[-1])
ax.axhline(17, color="#555555", linestyle="--")
ax.set(xlabel="Dealer cards held", ylabel="Best total", xticks=range(2, 6))
ax.legend()
plt.tight_layout()
plt.show()

## Splits and split-Ace restrictions

I can split any two ten-valued cards, even `10` and `K`. I can resplit non-Aces until I have four hands. Split Aces are different: each new hand receives exactly one card, cannot be resplit, and a resulting 21 pays 1:1.

In [ ]:
mixed_tens = BlackjackRound(Shoe.arranged(cards("10", "6", "K", "10", "2", "3")), 10)
print("10 + K legal actions:", [action.value for action in mixed_tens.legal_actions])

split_aces = BlackjackRound(
    Shoe.arranged(cards("A", "6", "A", "10", "K", "Q", "10")), 10
)
split_aces.act(PlayerAction.SPLIT)
for index, hand in enumerate(split_aces.public_state.player_hands, start=1):
    print(
        f"split-Ace hand {index}: cards={' '.join(str(card) for card in hand.cards)}, "
        f"natural=False, profit={hand.profit}"
    )

## Surrender, doubles, insurance, and exact payouts

I use `Fraction` throughout settlement, so half wagers and 3:2 payouts remain exact. Insurance costs half the original wager and earns 2:1 profit when the dealer has blackjack. Taking insurance alongside a player natural reproduces even money without a special action.

In [ ]:
surrender = BlackjackRound(Shoe.arranged(cards("10", "9", "6", "8")), 10)
surrender.act(PlayerAction.SURRENDER)

double = BlackjackRound(Shoe.arranged(cards("5", "6", "6", "10", "10", "10")), 10)
double.act(PlayerAction.DOUBLE)

natural = BlackjackRound(Shoe.arranged(cards("A", "6", "K", "10")), 10)

even_money = BlackjackRound(Shoe.arranged(cards("A", "A", "K", "10")), 10)
even_money.decide_insurance(InsuranceAction.TAKE)

settlement_examples = {
    "late surrender": surrender.public_state.settlement,
    "double win": double.public_state.settlement,
    "natural 3:2": natural.public_state.settlement,
    "natural + insurance": even_money.public_state.settlement,
}
profits = []
for label, settlement in settlement_examples.items():
    assert settlement is not None
    profits.append(float(settlement.total_profit))
    print(f"{label:20} net profit = {settlement.total_profit}")

fig, ax = plt.subplots(figsize=(7.5, 3.2))
ax.bar(
    settlement_examples.keys(),
    profits,
    color=["#e45756", "#54a24b", "#72b7b2", "#b279a2"],
)
ax.axhline(0, color="#333333", linewidth=1)
ax.set_ylabel("Exact net profit (shown as dollars)")
ax.tick_params(axis="x", rotation=20)
plt.tight_layout()
plt.show()

## Public information and the hidden hole card

I keep the authoritative state separate from the public snapshot. Before reveal, the internal log can record the hole card, but neither public events nor the model context can expose it. The model context partitions visible cards exactly once into history, current hand, and dealer upcard.

In [ ]:
visibility = BlackjackRound(Shoe.arranged(cards("5", "9", "6", "7", "10")), 10)
before = visibility.public_state
context = before.model_context
assert context is not None

print("public dealer cards: ", [str(card) for card in before.dealer_cards])
print(
    "internal dealer cards:",
    [str(card) for card in visibility.internal_state.dealer_cards],
)
print("model history:       ", [str(card) for card in context.history])
print("model current hand:  ", [str(card) for card in context.current_hand])
print("model dealer upcard: ", context.dealer_upcard)

visibility.act(PlayerAction.STAND)
after = visibility.public_state
print("after reveal:         ", [str(card) for card in after.dealer_cards])
print("visible once each:    ", [str(card) for card in after.visible_card_history])

## Deterministic replay

Here I replay the same shoe so I can inspect the exact same round twice. The replay stores both the physical deal order and the cut-card position; it does not depend on remembering how a particular random-number generator happened to shuffle.

In [ ]:
replay_data = Shoe.arranged(
    cards("10", "9", "7", "7", "4", "10"),
    cut_card_position=6,
).replay


def replay_round() -> BlackjackRound:
    game = BlackjackRound(Shoe.from_replay(replay_data), Fraction(10))
    game.act(PlayerAction.HIT)
    return game


first = replay_round()
second = replay_round()

print("same public state:", first.public_state == second.public_state)
print("same event log:   ", first.internal_state.events == second.internal_state.events)
print("cut position:     ", replay_data.cut_card_position)
print("deal order:       ", [str(card) for card in replay_data.deal_order])

seeded_a = Shoe.shuffled(2026)
seeded_b = Shoe.shuffled(2026)
print("same seeded shoe: ", seeded_a.replay == seeded_b.replay)

The examples give me a trusted boundary for the later notebooks: I can focus on tokenization and transformer mechanics while the engine continues to enforce legal actions, hidden information, deterministic cards, and exact money.